# Stage 6 — Concatenation Baselines (Human Lymph Node)

**Goal:** Test whether simple feature concatenation beats our neural model on A1/D1.

**Why:** Literature reports naive concat ARI 0.325 and variance-scaled concat 0.339
on human lymph node — both above our 0.25, with zero neural network. If concat
wins on our data too, the SAGE+MLP architecture is hurting, not helping.

**Method:** For A1/D1 (and E11-E18 for completeness):
- naive concat: standardize each modality, concatenate
- variance-scaled concat: scale each modality by 1/sqrt(total variance), then concat
- RNA-only, ADT-only / ATAC-only (ablation: which modality drives clusters?)
- Cluster: kmeans, GMM, spherical at declared k
- Also try PCA-32 on concatenated features (match embedding dim)

**Output:** `runs/s6-concat/`


In [ ]:
import os, sys, json, subprocess

for pkg in ["scikit-learn", "pandas", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
import scanpy as sc

print("imports ok")


In [ ]:
# ---------- GitHub push ----------
REPO = "/kaggle/working/my_research"
REPO_URL = "https://github.com/rifatahsanpul0k/my_research.git"
def sh(*args):
    r = subprocess.run(list(args), capture_output=True, text=True, cwd=REPO)
    return r
os.makedirs(REPO, exist_ok=True)
OUT = os.path.join(REPO, "runs", "s6-concat")
os.makedirs(OUT, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    import base64, shutil
    _sec = UserSecretsClient()
    _tok = _sec.get_secret('GITHUB_TOKEN')
    _b64 = base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    _GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {_b64}"]
    if not os.path.isdir(os.path.join(REPO, ".git")):
        if os.listdir(REPO):
            bak = REPO + "_stalebak"
            if os.path.exists(bak):
                shutil.rmtree(bak)
            shutil.move(REPO, bak)
            os.makedirs(REPO, exist_ok=True)
        subprocess.run(_GIT + ["clone", REPO_URL, REPO], check=True)
        sh("git", "config", "user.name", "kaggle-runner")
        sh("git", "config", "user.email", "kaggle-runner@local")
    os.makedirs(OUT, exist_ok=True)
    print("git ready")
except Exception as e:
    print("git setup issue:", str(e)[:200])

def git_push(msg):
    try:
        os.makedirs(OUT, exist_ok=True)
        sh("git", "add", "runs/s6-concat")
        r = sh("git", "diff", "--cached", "--quiet")
        if r.returncode == 0:
            print("nothing new to push")
            return
        sh("git", "commit", "-m", msg)
        r2 = subprocess.run(_GIT + ["pull", "--rebase", "origin", "main"],
                           capture_output=True, text=True, cwd=REPO)
        r3 = subprocess.run(_GIT + ["push", "origin", "main"],
                           capture_output=True, text=True, cwd=REPO)
        print("pushed")
    except Exception as e:
        print("push failed:", str(e)[:300])


In [ ]:
# ---------- Data loading (EXACT copy from stage5) ----------
MARKER = "10x_human_lymph_node_A1"
def find_data_root():
    for base in ["/kaggle/input/spatial-multiomics-6datasets-private",
                 "/kaggle/working/spatial-multiomics-6datasets-private",
                 "/kaggle/working"]:
        if os.path.isdir(os.path.join(base, MARKER)):
            return base
    return None

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    print("dataset not found, attempting download...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d",
                    "pulokpulok/spatial-multiomics-6datasets-private",
                    "-p", "/kaggle/working", "--unzip"], check=True)
    DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

DATASETS = {
    "A1":  dict(folder="10x_human_lymph_node_A1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "D1":  dict(folder="10x_human_lymph_node_D1", aux="adt",  gt_file="annotation.csv", gt_col="manual-anno"),
    "E11": dict(folder="Mouse_Brain_E11_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E13": dict(folder="Mouse_Brain_E13_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E15": dict(folder="Mouse_Brain_E15_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
    "E18": dict(folder="Mouse_Brain_E18_S1", aux="atac", gt_file="anno.csv", gt_col="cluster"),
}
DECLARED_K = {"A1": 10, "D1": 11, "E11": 8, "E13": 12, "E15": 12, "E18": 14}

def to_dense_small(X, max_elems=5e7):
    if hasattr(X, "toarray"):
        return X.toarray() if X.size <= max_elems else X
    return np.asarray(X)

def clr(X):
    X = np.asarray(X, dtype=np.float64)
    gm = np.exp(np.log1p(X).mean(axis=1, keepdims=True))
    return np.log1p(X / gm)

def load_dataset(ds_id):
    spec = DATASETS[ds_id]
    root = os.path.join(DATA_ROOT, spec["folder"])
    rna = sc.read_h5ad(os.path.join(root, "adata_RNA.h5ad"))
    rna.var_names_make_unique()
    sc.pp.normalize_total(rna, target_sum=1e4)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, n_top_genes=3000, flavor="seurat", inplace=True)
    Xh = rna[:, rna.var.highly_variable].X
    Xh = Xh.toarray() if hasattr(Xh, "toarray") else np.asarray(Xh)
    Xr = StandardScaler().fit_transform(PCA(n_components=64, random_state=0).fit_transform(Xh)).astype(np.float32)
    if spec["aux"] == "adt":
        aux = sc.read_h5ad(os.path.join(root, "adata_ADT.h5ad"))
        aux.var_names_make_unique()
        Xa = StandardScaler().fit_transform(clr(to_dense_small(aux.X))).astype(np.float32)
    else:
        aux = sc.read_h5ad(os.path.join(root, "adata_ATAC.h5ad"))
        aux.var_names_make_unique()
        X = aux.X.tocsr().astype(np.float64) if hasattr(aux.X, "tocsr") else sparse.csr_matrix(np.asarray(aux.X, dtype=np.float64))
        rs = np.asarray(X.sum(axis=1)).ravel() + 1e-9
        tf = sparse.diags(1.0 / rs).dot(X)
        df = np.asarray((X > 0).sum(axis=0)).ravel()
        idf = np.log(1 + X.shape[0] / (df + 1))
        lsi = TruncatedSVD(n_components=50, random_state=0).fit_transform(tf.multiply(idf))
        depth = np.log1p(np.asarray(X.sum(axis=1)).ravel())
        keep = [i for i in range(50) if abs(np.corrcoef(lsi[:, i], depth)[0, 1]) <= 0.75]
        Xa = StandardScaler().fit_transform(lsi[:, keep]).astype(np.float32)
    gt = pd.read_csv(os.path.join(root, spec["gt_file"]), index_col=0)
    print(f"{ds_id}: RNA {Xr.shape} AUX {Xa.shape}", flush=True)
    return dict(Xr=Xr, Xa=Xa, gt=np.asarray(gt[spec["gt_col"]]).astype(str))


In [ ]:
# ---------- Concat baselines ----------
def cluster(X, head, k, seed=0):
    np.random.seed(seed)
    if head == "kmeans":
        return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)
    elif head == "gmm":
        return GaussianMixture(n_components=k, n_init=3, random_state=seed).fit_predict(X)
    elif head == "spherical":
        Xn = normalize(X, norm="l2")
        return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Xn)
    raise ValueError(head)

# Neural bests for comparison
NEURAL_BEST = {"A1": 0.250, "D1": 0.250, "E11": 0.445, "E13": 0.436, "E15": 0.506, "E18": 0.323}

results = {}
fout = open(os.path.join(OUT, "concat_results.jsonl"), "a")

for ds_id in DATASETS:
    print(f"\n{'='*55}\nDataset: {ds_id}\n{'='*55}")
    data = load_dataset(ds_id)
    Xr, Xa, gt = data["Xr"], data["Xa"], data["gt"]
    k = DECLARED_K[ds_id]

    # Feature variants
    variants = {
        "naive_concat":    np.concatenate([Xr, Xa], axis=1),
        "rna_only":        Xr,
        "aux_only":        Xa,
    }
    # variance-scaled concat: scale each modality to unit total variance
    vr = Xr.var() * Xr.shape[1]
    va = Xa.var() * Xa.shape[1]
    variants["varscaled_concat"] = np.concatenate([Xr / np.sqrt(vr), Xa / np.sqrt(va)], axis=1)
    # pca32 versions
    for vname in ["naive_concat", "varscaled_concat"]:
        Xv = variants[vname]
        Xv = StandardScaler().fit_transform(Xv)
        variants[vname + "_pca32"] = PCA(n_components=32, random_state=0).fit_transform(Xv)

    ds_best = 0
    ds_best_cfg = ""
    for vname, Xv in variants.items():
        Xv = StandardScaler().fit_transform(Xv)
        for head in ["kmeans", "gmm", "spherical"]:
            aris = []
            for seed in [42, 1234, 2024]:
                labels = cluster(Xv, head, k, seed)
                aris.append(adjusted_rand_score(gt, labels))
            mean_ari = float(np.mean(aris))
            fout.write(json.dumps(dict(dataset=ds_id, variant=vname, head=head,
                                       ari=mean_ari)) + chr(10))
            fout.flush()
            if mean_ari > ds_best:
                ds_best = mean_ari
                ds_best_cfg = f"{vname}/{head}"
            print(f"  {vname:24} {head:9} ARI={mean_ari:.3f}")

    neural = NEURAL_BEST[ds_id]
    gap = neural - ds_best
    tag = "NEURAL WINS" if gap > 0.02 else ("CONCAT WINS" if gap < -0.02 else "TIE")
    print(f"\n  Best concat: {ds_best:.3f} ({ds_best_cfg})")
    print(f"  Neural best: {neural:.3f}  →  {tag} (gap {gap:+.3f})")
    results[ds_id] = {"best_concat_ari": ds_best, "best_concat_cfg": ds_best_cfg,
                      "neural_ari": neural, "gap": float(gap), "verdict": tag}

fout.close()

with open(os.path.join(OUT, "concat_summary.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*55}\nFINAL SUMMARY\n{'='*55}")
for ds_id, r in results.items():
    print(f"  {ds_id}: concat={r['best_concat_ari']:.3f} ({r['best_concat_cfg']}) vs neural={r['neural_ari']:.3f} → {r['verdict']}")

git_push("s6-concat: concatenation baseline results")
print("\nDone.")
